In this notebook, we build a baseline model to predict the frames win percentage by player1. In particular, the model predict the player with higher elo rating to be the winner. Then it predicts the winner to have the average win percentage p in the training set and 1-p for the loser. We record the mse on the test set.

In [1]:
import pandas as pd
import numpy as np

In [2]:
#Import the match data for the last 50 tournaments
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/4_Player_Data_Exploration/match_data_after_explorations.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   5058 non-null   object 
 1   player2                   5058 non-null   object 
 2   best_of                   5058 non-null   int64  
 3   player1_elo               5058 non-null   int64  
 4   player2_elo               5058 non-null   int64  
 5   elo_match_win_rate        5058 non-null   float64
 6   elo_frame_win_rate        5058 non-null   float64
 7   p1_matches_played         5058 non-null   int64  
 8   p1_matches_won            5058 non-null   int64  
 9   p1_frames_played          5058 non-null   int64  
 10  p1_frames_won             5058 non-null   int64  
 11  p2_matches_played         5058 non-null   int64  
 12  p2_matches_won            5058 non-null   int64  
 13  p2_frames_played          5058 non-null   int64  
 14  p2_frame

In [3]:
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id,p1_frames_win_rate,p2_frames_win_rate,p1_matches_win_rate,p2_matches_win_rate
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,17,5,0,0.0,1.000000,5808,0.515639,0.531250,0.524927,0.333333
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,430,2,5,1.0,0.285714,5808,0.173913,0.537687,0.000000,0.592593
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,147,5,3,0.0,0.625000,5808,0.503623,0.478563,0.474359,0.460808
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,97,5,2,0.0,0.714286,5808,0.530972,0.474479,0.565380,0.447876
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,649,2,5,1.0,0.285714,5808,0.417763,0.574481,0.352941,0.657339


In [4]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2)

In [5]:
# find the average win percentage of winner from training set.
winner_win_percs = []
for i in range(len(data_train)):
    match = data_train.iloc[i]
    match_result = match['match_result']
    win_perc = match['win_percentage']
    if match_result:
        winner_win_percs.append(1-win_perc)
    else: 
        winner_win_percs.append(win_perc)

print(winner_win_percs[:5])


[np.float64(0.8), np.float64(1.0), np.float64(1.0), np.float64(0.8), np.float64(0.7142857142857143)]


In [6]:
average_winner_win_perc = np.mean(winner_win_percs)
average_winner_win_perc


np.float64(0.7473027582755079)

In [7]:
#Making prediction on test set.
win_per_prediction = np.zeros(len(data_test))
for i in range(len(data_test)):
    match = data_test.iloc[i]
    if match['elo_match_win_rate']>=0.5:
        win_per_prediction[i] = average_winner_win_perc
    else: 
        win_per_prediction[i] = 1 - average_winner_win_perc

print(win_per_prediction[:5])


[0.74730276 0.25269724 0.74730276 0.25269724 0.25269724]


In [8]:
#Calculate rmse
from sklearn.metrics import root_mean_squared_error
score = root_mean_squared_error(win_per_prediction, data_test['win_percentage'].values)
print(score)

0.31230869425101954
